<a href="https://colab.research.google.com/github/guupiii/ESAA/blob/main/Gradient_XgBoost.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import pandas as pd
import numpy as np

In [5]:
!pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 10.2 MB/s eta 0:00:00


In [6]:
df = pd.read_csv('/content/drive/MyDrive/MLData/insurance.csv')

In [7]:
df = df.drop(columns=['region'])
df['charges'] = np.log1p(df['charges'])

In [8]:
from sklearn.model_selection import train_test_split
import numpy as np

idx = np.arange(len(df))

train_idx, test_idx = train_test_split(
    idx, test_size=0.2, random_state=42
)

In [9]:
df_xgb = pd.get_dummies(df, drop_first=True)

X_xgb = df_xgb.drop('charges', axis=1)
y = df_xgb['charges']

X_train_xgb = X_xgb.iloc[train_idx]
X_test_xgb = X_xgb.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

In [10]:
df_cat = df.copy()

X_cat = df_cat.drop('charges', axis=1)

X_train_cat = X_cat.iloc[train_idx]
X_test_cat = X_cat.iloc[test_idx]

In [11]:
from sklearn.model_selection import RandomizedSearchCV
from xgboost import XGBRegressor

xgb = XGBRegressor(random_state=42)

param_dist = {
    'n_estimators': [700, 900, 1100],
    'learning_rate': [0.01, 0.015, 0.02],
    'max_depth': [2, 3],
    'subsample': [0.75, 0.8, 0.85],
    'colsample_bytree': [0.75, 0.8, 0.85],
    'reg_alpha': [0, 0.05, 0.1],
    'reg_lambda': [2, 3, 4]
}

random_search = RandomizedSearchCV(
    xgb,
    param_distributions=param_dist,
    n_iter=30,
    scoring='neg_root_mean_squared_error',
    cv=3,
    random_state=42,
    n_jobs=-1
)

random_search.fit(X_train_xgb, y_train)

RandomizedSearchCV(cv=3,
                   estimator=XGBRegressor(base_score=None, booster=None,
                                          callbacks=None,
                                          colsample_bylevel=None,
                                          colsample_bynode=None,
                                          colsample_bytree=None, device=None,
                                          early_stopping_rounds=None,
                                          enable_categorical=False,
                                          eval_metric=None, feature_types=None,
                                          feature_weights=None, gamma=None,
                                          grow_policy=None,
                                          importance_type=None,
                                          interaction_constraint...
                                          multi_strategy=None,
                                          n_estimators=None, n_jobs=None,
                                          num_parallel_tree=None, ...),
                   n_iter=30, n_jobs=-1,
                   param_distributions={'colsample_bytree': [0.75, 0.8, 0.85],
                                        'learning_rate': [0.01, 0.015, 0.02],
                                        'max_depth': [2, 3],
                                        'n_estimators': [700, 900, 1100],
                                        'reg_alpha': [0, 0.05, 0.1],
                                        'reg_lambda': [2, 3, 4],
                                        'subsample': [0.75, 0.8, 0.85]},
                   random_state=42, scoring='neg_root_mean_squared_error')

In [12]:
best_xgb = random_search.best_estimator_

xgb_pred = best_xgb.predict(X_test_xgb)

In [13]:
import numpy as np
from sklearn.metrics import mean_squared_error

rmsle_xgb_rs = np.sqrt(
    mean_squared_error(
        y_test,
        xgb_pred
    )
)

print("random search XGBoost RMSLE:", rmsle_xgb_rs)

random search XGBoost RMSLE: 0.3489909784083606


In [14]:
from sklearn.ensemble import GradientBoostingRegressor

gbr = GradientBoostingRegressor(random_state=42)

gbr.fit(X_train_xgb, y_train)

GradientBoostingRegressor(random_state=42)

In [15]:
gb_pred = gbr.predict(X_test_xgb)

In [16]:
from sklearn.metrics import mean_squared_error
import numpy as np

gb_rmse = np.sqrt(mean_squared_error(y_test, gb_pred))
print("GB RMSLE:", gb_rmse)

GB RMSLE: 0.3512347059453921


In [17]:
from catboost import CatBoostRegressor
from sklearn.metrics import mean_squared_error
import numpy as np

cat = CatBoostRegressor(
    iterations=1000,
    learning_rate=0.03,
    depth=6,
    l2_leaf_reg=3,
    random_state=42,
    verbose=0
)

cat.fit(
    X_train_cat, y_train,
    eval_set=(X_test_cat, y_test),
    early_stopping_rounds=50,
    cat_features=['sex', 'smoker']
)

cat_pred = cat.predict(X_test_cat)

cat_rmse = np.sqrt(mean_squared_error(y_test, cat_pred))
print(f"CatBoost RMSLE: {cat_rmse:.4f}")

CatBoost RMSLE: 0.3509


# **앙상블**

In [18]:
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
import numpy as np

kf = KFold(n_splits=5, shuffle=True, random_state=42)

In [20]:
scores = []

for train_idx, val_idx in kf.split(X_train_xgb):

    # XGB / GB용 데이터
    X_tr_xgb = X_train_xgb.iloc[train_idx]
    X_val_xgb = X_train_xgb.iloc[val_idx]

    # CatBoost용 데이터
    X_tr_cat = X_train_cat.iloc[train_idx]
    X_val_cat = X_train_cat.iloc[val_idx]

    y_tr = y_train.iloc[train_idx]
    y_val = y_train.iloc[val_idx]

    # -------- 모델 학습 --------
    xgb = XGBRegressor(**random_search.best_params_, random_state=42)
    xgb.fit(X_tr_xgb, y_tr)

    gbr = GradientBoostingRegressor(random_state=42)
    gbr.fit(X_tr_xgb, y_tr)

    cat = CatBoostRegressor(
        iterations=1000,
        learning_rate=0.03,
        depth=6,
        l2_leaf_reg=3,
        random_state=42,
        verbose=0
    )

    cat.fit(
        X_tr_cat, y_tr,
        eval_set=(X_val_cat, y_val),
        early_stopping_rounds=50,
        cat_features=['sex', 'smoker']
    )

    # -------- 예측 --------
    xgb_pred = xgb.predict(X_val_xgb)
    gb_pred = gbr.predict(X_val_xgb)
    cat_pred = cat.predict(X_val_cat)

    # -------- 앙상블 --------
    final_pred = (
        0.5 * xgb_pred +
        0.3 * gb_pred +
        0.2 * cat_pred
    )

    score = np.sqrt(mean_squared_error(y_val, final_pred))
    scores.append(score)

print("Fold별 RMSLE:", scores)
print("평균 RMSLE:", np.mean(scores))

Fold별 RMSLE: [np.float64(0.4483540645141818), np.float64(0.3613412624828268), np.float64(0.33930832113063647), np.float64(0.3498453928237459), np.float64(0.40858479850106405)]
평균 RMSLE: 0.381486767890491
